In [ ]:
from nn import *
m = 64
def Cv(i=m, h=m, a=m, r=(15,0)):
    return Sequential(
        Linear(i, h, 0),
        T(Conv1d(h, h, sum(r)+1, padding=max(r), groups=h, bias=0)),
        F(lambda _:_[...,max(r[1]-r[0],0):min(r[1]-r[0],0)or None,:]if r[1]!=r[0]else _),
        Linear(h, i, 0),
        Res(
            RMSNorm(i, elementwise_affine=0),
            SwiGLU(i, a),
            Linear(a, i, 0),
        ),
    )
def At(i=m, h=int(m**.5), H=int(m**.5), a=m, g=int(m**.25), G=int(m**.25), r=(15,0)):
    return Sequential(
        Res(
            RMSNorm(i, elementwise_affine=0),
            Attn(i, h, H, g, G, r),
            Linear(G*g*(H or i), i, 0),
        ),
        Res(
            RMSNorm(i, elementwise_affine=0),
            SwiGLU(i, a),
            Linear(a, i, 0),
        ),
    )
def U(*_, i=m, h=m, s=2, S=-1):
    return Res(
        Linear(i, h, 0),
        T(MaxPool1d(s)),
        *_,
        T(ConvTranspose1d(h, h, s, s, output_padding=max(S,0), groups=h, bias=0)),
        F(lambda _:_[...,S:,:]if S>0 else functional.pad(_[...,:S,:],(0,0,-S,0))if S else _),
        Linear(h, i, 0),
    )
m = Sequential(
    Embedding(65536, m),
    Add(
        Sequential(
            At(),
            U(At(),
              U(At(),
                U(At()),
                At()),
              At()),
            At(),
            F(lambda _:functional.pad(_[...,:-1,:],(0,0,1,0))),
            Linear(m, m, 0),
        ),
        Sequential(
           At(r=(0,15)),
            U(At(r=(0,15)),
              U(At(r=(0,15)),
                U(At(r=(0,15)),S=1),
                At(r=(0,15)),S=1),
              At(r=(0,15)),S=1),
            At(r=(0,15)),
            F(lambda _:functional.pad(_[...,1:,:],(0,0,0,1))),
            Linear(m, m, 0),
        ),
    ),
    SiLU(),
    Linear(m, 65536),
).to('cuda',torch.bfloat16)
m = torch.compile(m, fullgraph=True, dynamic=False)
o = torch.optim.Adam(m.parameters(), 1/256)
a = torch.from_numpy(np.memmap('py.65536', mode='r'))
a = a.as_strided_((len(a)//1024-1,2048),(1024,1))
plot = Plot(2, 10, 10)

In [ ]:
for y in range(100000):
    t = a[torch.randint(len(a),[4])].to('cuda',torch.long)
    x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(65536,[t.numel()//2],device='cuda'))
    y = m(x)
    l = functional.cross_entropy(y.mT, t)
    (l*1024).backward()
    o.step()
    o.zero_grad()
    plot(l, lambda:(y.argmax(-1)==t).sum()/t.numel())